In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import seaborn as sns



## Load the dataset

In [ ]:
# Load the preprocessed DataFrame
df = pd.read_pickle("../data/interim/df_dtypes_fixed.pkl")

## Define variables

In [ ]:
X = df.drop(columns = ['is_fraud', 'fraud_type', 'transaction_id'])

We drop the **`fraud_type`** column, as it is not suitable for predicting fraudulent transactions and may introduce unnecessary complexity.
We also drop the **`transaction_id`** column, as it serves only as an identifier and does not provide predictive value.

In [ ]:
y = df['is_fraud']

## Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

We use **stratification** because the target class is heavily imbalanced, and we want to preserve the distribution of the fraud class.

## Handling missing values

We will use **User-Level Median Imputation** to handle missing values in the **`time_since_last_transaction`** column. This strategy preserves each user's typical behavior and minimizes the impact on fraud detection signals.  

The median will be computed **only on the training dataset**, and the same transformation will be applied to the test dataset.

In [ ]:
# Compute per-sender median on training data
user_medians = X_train.groupby('sender_account')['time_since_last_transaction'].median()

# Map medians to the training data
X_train['time_since_last_transaction'] = X_train['time_since_last_transaction'].fillna(
    X_train['sender_account'].map(user_medians)
)


Remaining missing values (users with all missing): 28432


In [ ]:
# Count remaining missing values
remaining_missing = X_train['time_since_last_transaction'].isna().sum()

# Calculate percentage
total_rows = len(X_train)
missing_percent = (remaining_missing / total_rows) * 100

print(f"Remaining missing values: {remaining_missing}")
print(f"Percentage of missing values: {missing_percent:.2f}%")

Remaining missing values: 28432
Percentage of missing values: 0.71%


Approximately **0.71% of the rows** belong to users for whom all values in the **`time_since_last_transaction`** column are missing. These missing values are likely **not due to data errors**; instead, they correspond to new users who appear only once in the dataset. In this case, leaving the values as NaN is appropriate.  

We will also adjust the missing-value flag to capture only users with entirely missing values in this column.  

For such users, leaving **`time_since_last_transaction`** as NaN preserves behavioral information instead of averaging it away. This missingness is **informative for fraud detection**, signaling to the model that the account is new or low-activity, which may correlate with certain types of fraud.

In [ ]:
# adjust the missing-value flag to capture only users with entirely missing history, not just individual transactions
X_train['time_since_last_transaction_missing'] = X_train['time_since_last_transaction'].isnull().astype(int)

Next, we apply the same logic to the test dataset, completing the missing-value imputation process.

In [ ]:
# Map training medians to test data
X_test['time_since_last_transaction'] = X_test['time_since_last_transaction'].fillna(
    X_test['sender_account'].map(user_medians)
)

# Add missingness flag for test data
X_test['time_since_last_transaction_missing'] = X_test['time_since_last_transaction'].isna().astype(int)

## Fixing problemetic values

0   transaction_id               str    
 1   timestamp                    str    
 2   sender_account               str    
 3   receiver_account             str    
 4   amount                       float64
 5   transaction_type             str    
 6   merchant_category            str    
 7   location                     str    
 8   device_used                  str    
 9   is_fraud                     bool   
 10  fraud_type                   str    
 11  time_since_last_transaction  float64
 12  spending_deviation_score     float64
 13  velocity_score               int64  
 14  geo_anomaly_score            float64
 15  payment_channel              str    
 16  ip_address                   str    
 17  device_hash                  str   

Our primary focus will be checking the numeric columns 'amount', 'time_since_last_transaction', 'spending_deviation_score', 'velocity_score' and 'geo_anomaly_score' 